# BlockGCN Fall Detection - 4-Stream Ensemble Training

**Train ALL 4 streams: Joint → Bone → Velocity → Bone-Velocity**

### ⚠️ Quan trọng
- Nếu gặp lỗi `ModuleNotFoundError`, hãy thử **Restart Kernel** sau khi chạy Cell 3 (Install Dependencies).
- Drive mount là tùy chọn.

### Workflow:
1. Setup environment & clone code
2. Train Stream 1: Joint
3. Train Stream 2: Bone
4. Train Stream 3: Velocity
5. Train Stream 4: Bone-Velocity
6. Save results (Drive hoặc Zip Download)

In [ ]:
# ============================================================
# CELL 1: Setup Environment (Drive Optional)
# ============================================================
print("📦 Setting up environment...")

USE_DRIVE = False
# Check GPU
import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: No GPU detected!")

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 2: Clone Code từ GitHub
# ============================================================
print("📥 Cloning code from GitHub...\n")

!git clone https://github.com/minhpd3112/FALL_DETECTION.git /content/FALL_DETECTION

print("\n✅ Code cloned!")
!ls -la /content/FALL_DETECTION/

In [ ]:
# ============================================================
# CELL 3: Install Dependencies (FIXED)
# ============================================================
print("📦 Installing BlockGCN dependencies...\n")

%cd /content/FALL_DETECTION/BlockGCN

# Install dependencies (force reinstall some)
%pip install -q pyyaml einops tensorboardX
%pip install -q git+https://github.com/mit-han-lab/torchpack.git
%pip install -q -e torchlight/

print("\n✅ Dependencies installed!")
print("⚠️  NẾU GẶP LỖI IMPORT Ở CELL DƯỚI -> HÃY RESTART KERNEL VÀ CHẠY LẠI TỪ CELL 4")

In [ ]:
# ============================================================
# CELL 4: Verify Data
# ============================================================
import numpy as np

data_dir = '/content/FALL_DETECTION/processed_data'
print("📊 Checking data files...\n")

try:
    X_train = np.load(f'{data_dir}/X_train.npy')
    y_train = np.load(f'{data_dir}/y_train.npy')
    print(f"✅ Train: {X_train.shape} - Labels: {y_train.shape}")
    print(f"Expected: (1437, 30, 17, 2)\n")
except Exception as e:
    print(f"❌ Lỗi data: {e}")
    print("Vui lòng kiểm tra lại repo GitHub hoặc quá trình clone.")

---
# 🔵 STREAM 1: JOINT
---

In [ ]:
# ============================================================
# STREAM 1/4: Train Joint Stream
# ============================================================
print("="*60)
print("🔵 [1/4] Training Joint Stream")
print("="*60)

%cd /content/FALL_DETECTION/BlockGCN

!python main.py \
    --config config/fall-detection/default.yaml \
    --phase train \
    --save-score True \
    --device 0 \
    --log-interval 10

print("\n✅ Joint stream completed!")

---
# 🟢 STREAM 2: BONE
---

In [ ]:
# ============================================================
# STREAM 2/4: Train Bone Stream
# ============================================================
print("="*60)
print("🟢 [2/4] Training Bone Stream")
print("="*60)

%cd /content/FALL_DETECTION/BlockGCN

!python main.py \
    --config config/fall-detection/bone.yaml \
    --phase train \
    --save-score True \
    --device 0 \
    --log-interval 10

print("\n✅ Bone stream completed!")

---
# 🟡 STREAM 3: VELOCITY
---

In [ ]:
# ============================================================
# STREAM 3/4: Train Velocity Stream
# ============================================================
print("="*60)
print("🟡 [3/4] Training Velocity Stream")
print("="*60)

%cd /content/FALL_DETECTION/BlockGCN

!python main.py \
    --config config/fall-detection/vel.yaml \
    --phase train \
    --save-score True \
    --device 0 \
    --log-interval 10

print("\n✅ Velocity stream completed!")

---
# 🔴 STREAM 4: BONE-VELOCITY
---

In [ ]:
# ============================================================
# STREAM 4/4: Train Bone-Velocity Stream
# ============================================================
print("="*60)
print("🔴 [4/4] Training Bone-Velocity Stream")
print("="*60)

%cd /content/FALL_DETECTION/BlockGCN

!python main.py \
    --config config/fall-detection/bone_vel.yaml \
    --phase train \
    --save-score True \
    --device 0 \
    --log-interval 10

print("\n✅ Bone-Velocity stream completed!")

---
# 💾 SAVE RESULTS
---

In [ ]:
# ============================================================
# Save Results (Drive or Local)
# ============================================================
from datetime import datetime
import shutil
import os

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Nếu Drive OK -> Copy sang Drive
if USE_DRIVE:
    save_dir = f'/content/drive/MyDrive/BlockGCN_Results/{timestamp}'
    print("="*60)
    print(f"💾 Saving All Results to Google Drive: {save_dir}")
    print("="*60)
    os.makedirs(save_dir, exist_ok=True)
    !cp -r /content/FALL_DETECTION/BlockGCN/work_dir {save_dir}/
    print("✅ Saved to Drive!")
else:
    print("⚠️ Drive not mounted. Skipping upload.")

# 2. Luôn nén Zip để Download thủ công (Backup)
print("\n" + "="*60)
print("📦 Zipping results for Manual Download...")
print("="*60)
!zip -r /content/BlockGCN_Results_FINAL.zip /content/FALL_DETECTION/BlockGCN/work_dir

print("\n✅ Zip file created: /content/BlockGCN_Results_FINAL.zip")
print("👉 Click chuột phải vào file này trong thanh bên trái -> Download")

---
# 📊 SUMMARY
---

In [ ]:
# ============================================================
# Check All Trained Models
# ============================================================
import glob

print("="*60)
print("📊 TRAINING SUMMARY")
print("="*60)

streams = ['joint', 'bone', 'velocity', 'bone_velocity']
for stream in streams:
    checkpoints = glob.glob(f'/content/FALL_DETECTION/BlockGCN/work_dir/fall_detection/{stream}/*/epoch_best.pt')
    if checkpoints:
        print(f"✅ {stream.upper()}: {checkpoints[0]}")
    else:
        print(f"❌ {stream.upper()}: NOT FOUND")

print("\n" + "="*60)
print("🎉 ALL 4 STREAMS COMPLETED!")
print("="*60)
print("\nNext: Use ensemble.py on these checkpoints")